<a href="https://colab.research.google.com/github/ABHIRAJAWAT-07/NLP_project/blob/new/kindle_project2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
data=pd.read_csv("/content/all_kindle_review.csv")
data.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [ ]:
data=data[['reviewText','rating']]

In [ ]:
data.shape

(12000, 2)

In [ ]:
##missing values
data.isnull().sum()

,0
reviewText,0
rating,0


In [ ]:
data['rating'].unique()

array([3, 5, 4, 2, 1])

In [ ]:
data['rating'].value_counts()

,count
rating,
5,3000
4,3000
3,2000
2,2000
1,2000


In [ ]:
##preprocessing and cleaning

In [ ]:
data['rating']=data['rating'].apply(lambda x:0 if x<3 else 1)

In [ ]:
data.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",1
1,Great short read. I didn't want to put it dow...,1
2,I'll start by saying this is the first of four...,1
3,Aggie is Angela Lansbury who carries pocketboo...,1
4,I did not expect this type of book to be in li...,1


In [ ]:
data['rating'].unique()

array([1, 0])

In [ ]:
data['rating'].value_counts()

,count
rating,
1,8000
0,4000


In [ ]:
##lower all the cases
data['reviewText']=data['reviewText'].str.lower()

In [ ]:
data.head()

,reviewText,rating
0,"jace rankin may be short, but he's nothing to ...",1
1,great short read. i didn't want to put it dow...,1
2,i'll start by saying this is the first of four...,1
3,aggie is angela lansbury who carries pocketboo...,1
4,i did not expect this type of book to be in li...,1


In [ ]:
import re
import nltk
from bs4 import BeautifulSoup
from nltk.corpus import stopwords

# Download stopwords dataset
nltk.download('stopwords', quiet=True)

## Removing special characters
data['reviewText']=data['reviewText'].apply(lambda x:re.sub('[^a-z A-z 0-9-]+', '',str(x)))

## Remove the stopwords
stop_words = set(stopwords.words('english'))
data['reviewText']=data['reviewText'].apply(lambda x:" ".join([y for y in str(x).split() if y not in stop_words]))

## Remove url
data['reviewText']=data['reviewText'].apply(lambda x: re.sub(r'(http|https|ftp|ssh)://([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])?', '' , str(x)))

## Remove html tags
data['reviewText']=data['reviewText'].apply(lambda x: BeautifulSoup(str(x), 'lxml').get_text())

## Remove any additional spaces
data['reviewText']=data['reviewText'].apply(lambda x: " ".join(str(x).split()))

In [ ]:
data.head()

,reviewText,rating
0,jace rankin may short hes nothing mess man hau...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four books wasnt expect...,1
3,aggie angela lansbury carries pocketbooks inst...,1
4,expect type book library pleased find price right,1


In [ ]:
##Lemmatizer
from nltk.stem import WordNetLemmatizer

In [ ]:
lemmatizer=WordNetLemmatizer()

In [ ]:
def lemmatize_words(text):
  return " ".join([lemmatizer.lemmatize(word) for word in text.split()])

In [ ]:
data['reviewText']=data['reviewText'].apply(lambda x: " ".join(str(x).split()))

In [ ]:
data.head()

,reviewText,rating
0,jace rankin may short hes nothing mess man hau...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four books wasnt expect...,1
3,aggie angela lansbury carries pocketbooks inst...,1
4,expect type book library pleased find price right,1


In [ ]:
##train test split
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(data['reviewText'],data['rating'],test_size=0.20)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
bow=CountVectorizer()
X_train_bow=bow.fit_transform(X_train).toarray()
X_test_bow=bow.transform(X_test)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer()
X_train_tfidf=tfidf.fit_transform(X_train).toarray()
X_test_tfodf=tfidf.transform(X_test)

In [ ]:
X_train_bow

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(9600, 39067))

In [ ]:
from sklearn.naive_bayes import GaussianNB
nb_model_bow=GaussianNB().fit(X_train_bow,y_train)
nb_model_tfidf=GaussianNB().fit(X_train_tfidf,y_train)

In [ ]:
from sklearn.metrics import confusion_matrix,accuracy_score,classification_report

In [ ]:
y_pred_bow=nb_model_bow.predict(X_test_bow.toarray())

In [ ]:
y_pred_tfidf = nb_model_tfidf.predict(X_test_tfodf.toarray())

# Evaluate TF-IDF Naive Bayes Model
print("Accuracy Score:", accuracy_score(y_test,y_pred_bow))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_bow))
print("\nClassification Report:\n", classification_report(y_test, y_pred_tfidf))

Accuracy Score: 0.5804166666666667

Confusion Matrix:
 [[468 293]
 [714 925]]

Classification Report:
               precision    recall  f1-score   support

           0       0.39      0.59      0.47       761
           1       0.75      0.57      0.65      1639

    accuracy                           0.58      2400
   macro avg       0.57      0.58      0.56      2400
weighted avg       0.64      0.58      0.59      2400



In [ ]:
print("tfidf Accuracy Score:", accuracy_score(y_test,y_pred_tfidf))

tfidf Accuracy Score: 0.5783333333333334
